In [1]:
import httpx
from brollm import BaseContract
from typing import Any

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen3.8:latest"

def input_fn(
    system_prompt: str,
    messages: list[dict],
    tool_registry: list[dict] | None = None,
) -> dict:
    payload = {
        "model": MODEL,
        "messages": [{"role": "system", "content": system_prompt}, *messages],
        "stream": False,
    }
    if tool_registry:
        payload["tools"] = tool_registry
    resp = httpx.post(OLLAMA_URL, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()

def output_fn(response: dict) -> Any:
    return response

llm = BaseContract(input_fn=input_fn, output_fn=output_fn)


In [5]:
tool_registry = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    },
]

reply = llm(
    system_prompt="You are a helpful assistant.",
    messages=[{"role": "user", "content": "What's the weather in Paris?"}],
    tool_registry=tool_registry,
)

reply


{'model': 'qwen3.8:latest',
 'created_at': '2026-09-12T04:42:36.362601Z',
 'message': {'role': 'assistant',
  'content': '',
  'thinking': 'The user is asking about the weather in Paris. I have a tool available to get the weather for a city. Let me call the get_weather function with "Paris" as the city.',
  'tool_calls': [{'id': 'call_1jxsbvf8',
    'function': {'index': 0,
     'name': 'get_weather',
     'arguments': {'city': 'Paris'}}}]},
 'done': True,
 'done_reason': 'stop',
 'total_duration': 3174572800,
 'load_duration': 7248200,
 'prompt_eval_count': 281,
 'prompt_eval_cached_count': 0,
 'prompt_eval_duration': 644194000,
 'eval_count': 67,
 'eval_duration': 2403390000}

In [6]:
reply = llm(
    system_prompt="You are a helpful assistant.",
    messages=[{"role": "user", "content": "What's 12 * 7?"}],
    tool_registry=tool_registry,
)

reply

{'model': 'qwen3.8:latest',
 'created_at': '2026-09-12T04:42:40.8990434Z',
 'message': {'role': 'assistant',
  'content': '12 × 7 = **84**',
  'thinking': "The user is asking a simple math question: 12 * 7. This doesn't require any tool calls.\n\n12 * 7 = 84"},
 'done': True,
 'done_reason': 'stop',
 'total_duration': 2362211900,
 'load_duration': 2900200,
 'prompt_eval_count': 283,
 'prompt_eval_cached_count': 0,
 'prompt_eval_duration': 639798000,
 'eval_count': 48,
 'eval_duration': 1715440000}

In [7]:
tool_registry = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "Convert an amount from one currency to another",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number"},
                    "from_currency": {"type": "string"},
                    "to_currency": {"type": "string"},
                },
                "required": ["amount", "from_currency", "to_currency"],
            },
        },
    },
]

messages = [
    {
        "role": "user",
        "content": "What's the weather in Tokyo, and how much is 100 USD in JPY?",
    }
]

reply = llm(
    system_prompt="You are a helpful assistant.",
    messages=messages,
    tool_registry=tool_registry,
)

reply['message']


{'role': 'assistant',
 'content': '',
 'thinking': 'The user wants two things:\n1. Weather in Tokyo\n2. Convert 100 USD to JPY\n\nThese are independent calls, so I can make them simultaneously.',
 'tool_calls': [{'id': 'call_ux34n819',
   'function': {'index': 0,
    'name': 'get_weather',
    'arguments': {'city': 'Tokyo'}}},
  {'id': 'call_ikxjtazc',
   'function': {'index': 1,
    'name': 'convert_currency',
    'arguments': {'amount': 100,
     'from_currency': 'USD',
     'to_currency': 'JPY'}}}]}

In [8]:
reply['message']['tool_calls']

[{'id': 'call_ux34n819',
  'function': {'index': 0,
   'name': 'get_weather',
   'arguments': {'city': 'Tokyo'}}},
 {'id': 'call_ikxjtazc',
  'function': {'index': 1,
   'name': 'convert_currency',
   'arguments': {'amount': 100,
    'from_currency': 'USD',
    'to_currency': 'JPY'}}}]

In [9]:
message = reply["message"]  # the assistant turn containing tool_calls
messages.append(message)    # keep it in history so the model sees its own calls

weather_result = "22°C and sunny in Tokyo"
conversion_result = "100 USD = 15,050 JPY"

messages.append({
    "role": "tool",
    "tool_name": "get_weather",
    "content": weather_result,
})
messages.append({
    "role": "tool",
    "tool_name": "convert_currency",
    "content": conversion_result,
})

final = llm(
    system_prompt="You are a helpful assistant.",
    messages=messages,
    # tool_registry=tool_registry,
)
# 548, 508 with tool_registry
final

{'model': 'qwen3.8:latest',
 'created_at': '2026-09-12T04:42:55.7128181Z',
 'message': {'role': 'assistant',
  'content': "Here's what you need:\n\n- **Weather in Tokyo:** 22°C and sunny ☀️\n- **100 USD in JPY:** 15,050 JPY (at the current exchange rate)\n\nLet me know if you need anything else!",
  'thinking': 'I have both results. Let me present them clearly.'},
 'done': True,
 'done_reason': 'stop',
 'total_duration': 4837951400,
 'load_duration': 3259300,
 'prompt_eval_count': 205,
 'prompt_eval_cached_count': 0,
 'prompt_eval_duration': 630904000,
 'eval_count': 74,
 'eval_duration': 4112346999}

In [10]:
messages.append(final['message'])

In [11]:
messages

[{'role': 'user',
  'content': "What's the weather in Tokyo, and how much is 100 USD in JPY?"},
 {'role': 'assistant',
  'content': '',
  'thinking': 'The user wants two things:\n1. Weather in Tokyo\n2. Convert 100 USD to JPY\n\nThese are independent calls, so I can make them simultaneously.',
  'tool_calls': [{'id': 'call_ux34n819',
    'function': {'index': 0,
     'name': 'get_weather',
     'arguments': {'city': 'Tokyo'}}},
   {'id': 'call_ikxjtazc',
    'function': {'index': 1,
     'name': 'convert_currency',
     'arguments': {'amount': 100,
      'from_currency': 'USD',
      'to_currency': 'JPY'}}}]},
 {'role': 'tool',
  'tool_name': 'get_weather',
  'content': '22°C and sunny in Tokyo'},
 {'role': 'tool',
  'tool_name': 'convert_currency',
  'content': '100 USD = 15,050 JPY'},
 {'role': 'assistant',
  'content': "Here's what you need:\n\n- **Weather in Tokyo:** 22°C and sunny ☀️\n- **100 USD in JPY:** 15,050 JPY (at the current exchange rate)\n\nLet me know if you need anythi

In [12]:
final

{'model': 'qwen3.8:latest',
 'created_at': '2026-09-12T04:42:55.7128181Z',
 'message': {'role': 'assistant',
  'content': "Here's what you need:\n\n- **Weather in Tokyo:** 22°C and sunny ☀️\n- **100 USD in JPY:** 15,050 JPY (at the current exchange rate)\n\nLet me know if you need anything else!",
  'thinking': 'I have both results. Let me present them clearly.'},
 'done': True,
 'done_reason': 'stop',
 'total_duration': 4837951400,
 'load_duration': 3259300,
 'prompt_eval_count': 205,
 'prompt_eval_cached_count': 0,
 'prompt_eval_duration': 630904000,
 'eval_count': 74,
 'eval_duration': 4112346999}

# Test load skill idea

In [15]:
import httpx
from brollm import BaseContract
from typing import Any
from pathlib import Path
import subprocess
import sys
from brocode.skills.utils import list_skills, load_skill_body, load_script
from brocode.utils.tool_management import load_tool_schema, to_args

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen3.8:latest"

def input_fn(
    system_prompt: str,
    messages: list[dict],
    tool_registry: list[dict] | None = None,
) -> dict:
    payload = {
        "model": MODEL,
        "messages": [{"role": "system", "content": system_prompt}, *messages],
        "stream": False,
    }
    if tool_registry:
        payload["tools"] = tool_registry
    resp = httpx.post(OLLAMA_URL, json=payload, timeout=180)
    resp.raise_for_status()
    return resp.json()

def output_fn(response: dict) -> Any:
    return response

def register_tool(tools:list)->list:
    registered_tools = []
    for tool in tools:
        _tool = {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        arg.name: {
                            "type": arg.type,
                            "description": arg.description
                        } for arg in tool.args
                    },
                    "required": [arg.name for arg in tool.args if arg.required]
                }
            }
        }
        registered_tools.append(_tool)
    return registered_tools

llm = BaseContract(input_fn=input_fn, output_fn=output_fn)


In [17]:
ROOT = Path.cwd().resolve().parent
SKILLS = list_skills(path=ROOT)

In [18]:
skill = [skill for skill in SKILLS if skill.name=='read-file'][0]
scripts = load_script(skill)
tools = [load_tool_schema(s) for s in scripts]
register_tool(tools)

[{'type': 'function',
  'function': {'name': 'read_directory',
   'description': 'Use this function when the person asks for reading all files in a directory',
   'parameters': {'type': 'object',
    'properties': {'path': {'type': 'string',
      'description': 'a directory path'}},
    'required': ['path']}}},
 {'type': 'function',
  'function': {'name': 'read_file',
   'description': 'Use this function when the person asks for reading only a single file',
   'parameters': {'type': 'object',
    'properties': {'path': {'type': 'string', 'description': 'a file path'}},
    'required': ['path']}}}]

In [19]:
TOOL_REGISTRY = [
    {
        "type": "function",
        "function": {
            "name": "load_skill",
            "description": (
                "Load the full instructions for a registered skill by name. "
                "Call this only when the current task clearly matches that "
                "skill's description."
            ),
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string"}},
                "required": ["name"],
            },
        },
    },
]
_TOOL_REGISTRY = [
    {
        "type": "function",
        "function": {
            "name": "load_reference",
            "description": (
                "Load the full contents of one reference document belonging to "
                "an already-loaded skill. Call this only when that skill's "
                "instructions point you to a specific reference file for more "
                "detail — don't call it speculatively."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The reference file's path exactly as shown in the skill's instructions, e.g. 'references/aws.md'."
                    }
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "ask_followup_question",
            "description": (
                "Signal that you need the user to answer something before you can "
                "continue. Write the actual question as your normal response "
                "content, then call this tool with no arguments to pause the turn "
                "and wait for their reply."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },

]

In [20]:
# system_prompt = (
#     "You are a helpful assistant. "
#     "Whenever your response asks the user a question and you need their answer "
#     "before continuing, you must call ask_followup_question — do not just write "
#     "the question as plain text alone."
# )

system_prompt = (
    "You are a helpful assistant. "
    "Whenever you need the user to answer something before continuing, your response "
    "must contain BOTH: (1) the actual question written out in your normal content, "
    "and (2) a call to ask_followup_question. Never call ask_followup_question with "
    "empty content — the user only sees what's in content, the tool call alone shows "
    "them nothing."
)


In [21]:
SKILL_DIR = ROOT / "skills"
def load_reference(path:str)->str:
    path = str(Path(path))
    references = list(SKILL_DIR.rglob(path))
    reference = [r for r in references if path in str(r)][0]
    if reference and reference.is_file():
        return reference.read_text()
    raise FileNotFoundError(f"Reference file not found: {path}")

In [22]:
def run_turn(messages):
    turn_alive = True
    turn_messages = messages.copy()
    loaded_skill = []
    _TOOLS = []
    try:
        while turn_alive:
            if not loaded_skill:
                turn_messages[-1]['content'] += f"""\nSKILLS:\n{"\n".join([f"- {s.name}: {s.description}" for s in SKILLS])}"""
            response = llm(system_prompt=system_prompt,
                messages=turn_messages,
                tool_registry=_TOOL_REGISTRY if loaded_skill else TOOL_REGISTRY
            )
            turn_messages.append(response['message'])
            if 'tool_calls' in response['message']:
                tool_list = response['message']['tool_calls']
                for t in tool_list:
                    name = t['function']['name']
                    arg = t['function']['arguments']

                    if name == 'ask_followup_question':
                        follow_up_question = response['message'].get('content', '').strip()
                        if not follow_up_question:
                            content = "you called ask_followup_question with no visible question — please ask again, including the question text"
                            turn_messages.append({"role": "tool", "tool_name": name, "content": content})
                        else:
                            content = input(follow_up_question)
                            turn_messages.append({"role": "user", "content": content})

                    elif (name == 'load_skill') and (arg['name'] not in loaded_skill):
                        skill = [s for s in SKILLS if s.name==arg['name']][0]
                        skill_body = load_skill_body(skill)
                        turn_messages.append({"role": "tool", "tool_name": name, "content": skill_body})
                        loaded_skill.append(arg['name'])
                        scripts = load_script(skill)
                        tools = [load_tool_schema(s) for s in scripts]
                        _TOOLS.extend(tools)
                        _tools = register_tool(tools)
                        _TOOL_REGISTRY.extend(_tools)

                    elif name == 'load_reference':
                        content = load_reference(arg['path'])
                        turn_messages.append({"role": "tool", "tool_name": name, "content": content})

                    else:
                        tool = [t for t in _TOOLS if t.name==name][0]
                        tool_path = tool.path
                        result = subprocess.run(
                            [sys.executable, str(tool_path), *to_args(arg)],
                            capture_output=True, text=True
                        )
                        turn_messages.append({"role": "tool", "tool_name": name, "content": result.stdout})
            else:                
                break
    except Exception as e:
        print(f"Error occurred: {e}")

    finally:
        return turn_messages, _TOOL_REGISTRY


In [24]:
# content = "read README.md for me."
content = "read everything under memory/* for me."
# content = "I failed the exam today and I feel a bit blue."
messages = [{"role": "user", "content": content}]
turn_messages, _ = run_turn(messages)

In [25]:
turn_messages

[{'role': 'user',
  'content': 'read everything under memory/* for me.\nSKILLS:\n- read-file: use this skill only when the person wanna read any type of file\n- tell-jokes: Tell a joke when the user asks for one, or when they seem sad, frustrated, or stressed and could use a laugh'},
 {'role': 'assistant',
  'content': '',
  'thinking': 'The user wants to read files under memory/*. This matches the read-file skill. Let me load it.',
  'tool_calls': [{'id': 'call_vfgaikqr',
    'function': {'index': 0,
     'name': 'load_skill',
     'arguments': {'name': 'read-file'}}}]},
 {'role': 'tool',
  'tool_name': 'load_skill',
  'content': "# Read file\n\nwhen invoked, read and return the file's content"},
 {'role': 'assistant',
  'content': '',
  'thinking': 'The user wants everything under memory/*. I should list the directory first to see what\'s there, then read the files. But the skill says read and return the file\'s content. Let me read the directory.\n\nActually, "memory/*" is a directo

In [26]:
_

[{'type': 'function',
  'function': {'name': 'load_reference',
   'description': "Load the full contents of one reference document belonging to an already-loaded skill. Call this only when that skill's instructions point you to a specific reference file for more detail — don't call it speculatively.",
   'parameters': {'type': 'object',
    'properties': {'path': {'type': 'string',
      'description': "The reference file's path exactly as shown in the skill's instructions, e.g. 'references/aws.md'."}},
    'required': ['path']}}},
 {'type': 'function',
  'function': {'name': 'ask_followup_question',
   'description': 'Signal that you need the user to answer something before you can continue. Write the actual question as your normal response content, then call this tool with no arguments to pause the turn and wait for their reply.',
   'parameters': {'type': 'object', 'properties': {}, 'required': []}}},
 {'type': 'function',
  'function': {'name': 'read_directory',
   'description': '

In [27]:
turn_messages[-1]

{'role': 'assistant',
 'content': "Here's everything under `memory/`:\n\n**file1.md**\n> I like the way you look at me.\n\n**file2.md**\n> Cat is the best.",
 'thinking': "I'm ready to present the contents of both files."}

In [28]:
print(turn_messages[-1]['content'])

Here's everything under `memory/`:

**file1.md**
> I like the way you look at me.

**file2.md**
> Cat is the best.


# Breakpoint

In [ ]:
from brocode.utils.tool_management import load_tool_schema
from pathlib import Path
SCRIPT_DIR = Path.cwd().resolve().parent / "skills" / "read-file" / "scripts"
tools = [load_tool_schema(SCRIPT_DIR / filename) for filename in ["read_file.py", "read_directory.py"]]
registered_tools = register_tool(tools)

content = """\
Read what is in skills/*
"""

reply = llm(
    system_prompt="You are a helpful assistant.",
    messages=[{"role": "user", "content": content}],
    tool_registry=registered_tools,
)

reply


{'model': 'qwen3.8:latest',
 'created_at': '2026-09-12T14:30:21.4533463Z',
 'message': {'role': 'assistant',
  'content': '',
  'thinking': "The user is asking me to read all files in the skills/* directory. Let me first read the directory to see what's there.",
  'tool_calls': [{'id': 'call_ozuhi18y',
    'function': {'index': 0,
     'name': 'read_directory',
     'arguments': {'path': 'skills'}}}]},
 'done': True,
 'done_reason': 'stop',
 'total_duration': 3273432900,
 'load_duration': 3661600,
 'prompt_eval_count': 367,
 'prompt_eval_cached_count': 0,
 'prompt_eval_duration': 917229000,
 'eval_count': 55,
 'eval_duration': 2347627000}

In [3]:
from brocode.skills.utils import _load_skill
from functools import partial
from pathlib import Path

ROOT = Path.cwd().resolve().parent
SKILLS_DIR = ROOT / "skills"
load_skill = partial(_load_skill, path=SKILLS_DIR)

TOOLS = {
    'load_skill': load_skill
}

In [17]:
SKILLS = [load_skill(s.name) for s in list(SKILLS_DIR.glob("*"))]
SKILL_INSTRUCTION = [f"- {s.name}: {s.description}" for s in SKILLS]
SKILL_INSTRUCTION

['- read-file: use this skill only when the person wanna read any type of file']

In [18]:
tool_registry = [
    {
        "type": "function",
        "function": {
            "name": "load_skill",
            "description": (
                "Load the full instructions for a registered skill by name. "
                "Call this only when the current task clearly matches that "
                "skill's description."
            ),
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string"}},
                "required": ["name"],
            },
        },
    },
]

template = """\
Select the most relevant skill for the following task: 
{skills}

User request:
{request}
"""

content = template.format(
    skills="\n".join(SKILL_INSTRUCTION),
    request="I wanna know what's inside main.py",
)


reply = llm(
    system_prompt="You are a helpful assistant.",
    messages=[{"role": "user", "content": content}],
    tool_registry=tool_registry,
)

reply


{'model': 'qwen3.8:latest',
 'created_at': '2026-09-12T09:43:07.9821501Z',
 'message': {'role': 'assistant',
  'content': '',
  'thinking': 'The user wants to know what\'s inside main.py - this is a request to read a file. The "read-file" skill matches this task since the user wants to read a file.',
  'tool_calls': [{'id': 'call_v7xbbhom',
    'function': {'index': 0,
     'name': 'load_skill',
     'arguments': {'name': 'read-file'}}}]},
 'done': True,
 'done_reason': 'stop',
 'total_duration': 18390711600,
 'load_duration': 14774051900,
 'prompt_eval_count': 333,
 'prompt_eval_cached_count': 0,
 'prompt_eval_duration': 685334000,
 'eval_count': 68,
 'eval_duration': 2927417000}

In [20]:
content = template.format(
    skills="\n".join(SKILL_INSTRUCTION),
    request="Today is hotter than yesterday, what's the weather like?",
)


reply = llm(
    system_prompt="You are a helpful assistant.",
    messages=[{"role": "user", "content": content}],
    tool_registry=tool_registry,
)

reply


{'model': 'qwen3.8:latest',
 'created_at': '2026-09-12T09:47:09.6604735Z',
 'message': {'role': 'assistant',
  'content': 'I appreciate the question, but I don\'t actually have access to real-time weather data or information about your location. I can\'t tell you today\'s or yesterday\'s temperature.\n\nHere are a few ways you can check the weather:\n\n- **Search your browser** for "weather in [your city]"\n- **Use a weather app** on your phone (like Weather, AccuWeather, or WeatherKit)\n- **Ask a voice assistant** (Siri, Alexa, Google Assistant) "What\'s the weather today?"\n\nIs there anything else I can help you with?',
  'thinking': "The user is asking about the weather. Let me consider the available skill:\n\n- read-file: use this skill only when the person wants to read any type of file\n\nThe user is asking about the weather, not reading a file. The read-file skill is specifically for reading files, which doesn't match this request at all.\n\nSo none of the available skills are 

In [38]:
from pathlib import Path
ROOT = Path.cwd().resolve().parent
ROOT

WindowsPath('D:/brocode')

In [49]:
test = 'references/*'
clean_path = test.rstrip("/*")
list(ROOT.rglob(clean_path))

[WindowsPath('D:/brocode/skills/tell-jokes/references')]

In [51]:
list(ROOT.rglob("references"))

[WindowsPath('D:/brocode/skills/tell-jokes/references')]